<a href="https://colab.research.google.com/github/iiiyahiya111/Project/blob/main/inspect_dataset_pairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/Potato_Severity_Dataset.zip"
extract_path = "/content/drive/MyDrive/Potato_Severity_Dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print("Dataset location:", extract_path)

Extraction completed!
Dataset location: /content/drive/MyDrive/Potato_Severity_Dataset


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Potato_Severity_Dataset"):
    level = root.replace("/content/drive/MyDrive/Potato_Severity_Dataset", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:5]:
        print(f"{indent}    {file}")

Potato_Severity_Dataset/
    manifest.csv
    test/
        images/
            311_Potato_Healthy_train_JPG.rf.f6d76e39abfc5329d59b1beaad815444.jpg
            36_Potato_Late_Blight_train_JPG.rf.ef6447e8b911201767a75e0622b8ebeb.jpg
            238_Potato_Healthy_train_JPG.rf.229ed2c5c6f350eed8380b6ba56d4516.jpg
            318_Potato_Healthy_train_JPG.rf.6278a2e9d4d5e57e87d0407b8fb93c64.jpg
            116_Potato_Late_Blight_train_JPG.rf.58749f79d3ad0853a5a9a30062bccb5e.jpg
        lesion_masks/
            389_Potato_Healthy_train_JPG.rf.48902f456fb48395a2e119a4ce314e5d.png
            21_Potato_Late_Blight_val_V2_JPG.rf.015ccbcdcc1d1535533f24dcc7fe436a.png
            380_Potato_Healthy_train_JPG.rf.283bbc7b5a793595f7a9ef6790a30b53.png
            270_Potato_Late_Blight_train_JPG.rf.464bfe478ec895f3c58d6fc4fd7e498e.png
            444_Potato_Late_Blight_train_JPG.rf.258468e9e8336d7088b0e04844b4e893.png
    train/
        images/
            218_Potato_Late_Blight_train_JPG.rf.a57d17

# **Main**

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt


# ============================================================
# DATASET ROOT
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/Potato_Severity_Dataset"
)

SPLITS = ["train", "valid", "test"]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MASK_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


# ============================================================
# BUILD FILE MAP
# ============================================================

def build_file_map(folder, allowed_extensions):
    file_map = {}

    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in allowed_extensions:
            file_map[p.stem] = p

    return file_map


# ============================================================
# SHOW ORIGINAL IMAGE
# ============================================================

def show_original(image_path, title):

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Cannot read image:\n{image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# ============================================================
# SHOW IMAGE + MASK
# ============================================================

def show_mask_verification(image_path, mask_path, title):

    image = cv2.imread(str(image_path))
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError(f"Cannot read image:\n{image_path}")

    if mask is None:
        raise ValueError(f"Cannot read mask:\n{mask_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Check dimensions
    if image.shape[:2] != mask.shape[:2]:
        raise ValueError(
            f"\nSIZE MISMATCH\n"
            f"Image: {image.shape[:2]}\n"
            f"Mask : {mask.shape[:2]}\n"
            f"\nImage: {image_path}\n"
            f"Mask : {mask_path}"
        )

    # Binary mask
    binary_mask = mask > 0

    # Create red overlay
    overlay = image.copy()

    red = image.copy()
    red[:, :, 0] = 255
    red[:, :, 1] = 0
    red[:, :, 2] = 0

    alpha = 0.45

    overlay[binary_mask] = (
        (1 - alpha) * image[binary_mask]
        + alpha * red[binary_mask]
    ).astype("uint8")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("Image + Lesion Mask")
    axes[1].axis("off")

    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Lesion Mask")
    axes[2].axis("off")

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()


# ============================================================
# INSPECT ONE SPLIT
# ============================================================

def inspect_split(split):

    image_dir = DATASET_ROOT / split / "images"
    mask_dir = DATASET_ROOT / split / "lesion_masks"

    print("\n" + "=" * 70)
    print(f"CHECKING: {split.upper()}")
    print("=" * 70)

    if not image_dir.exists():
        print(f"Image folder not found:\n{image_dir}")
        return

    if not mask_dir.exists():
        print(f"Lesion mask folder not found:\n{mask_dir}")
        return

    image_map = build_file_map(
        image_dir,
        IMAGE_EXTENSIONS
    )

    mask_map = build_file_map(
        mask_dir,
        MASK_EXTENSIONS
    )

    image_stems = set(image_map.keys())
    mask_stems = set(mask_map.keys())

    # --------------------------------------------------------
    # Pairing check
    # --------------------------------------------------------

    missing_masks = sorted(
        image_stems - mask_stems
    )

    orphan_masks = sorted(
        mask_stems - image_stems
    )

    paired_stems = sorted(
        image_stems & mask_stems
    )

    print(f"Images          : {len(image_map)}")
    print(f"Lesion masks    : {len(mask_map)}")
    print(f"Paired          : {len(paired_stems)}")
    print(f"Missing masks   : {len(missing_masks)}")
    print(f"Orphan masks    : {len(orphan_masks)}")

    # --------------------------------------------------------
    # Print pairing errors
    # --------------------------------------------------------

    if missing_masks:

        print("\nImages WITHOUT matching masks:")

        for stem in missing_masks:
            print("  ", stem)

    if orphan_masks:

        print("\nMasks WITHOUT matching images:")

        for stem in orphan_masks:
            print("  ", stem)

    # --------------------------------------------------------
    # Visual inspection
    # --------------------------------------------------------

    print("\nStarting visual inspection...")

    print(
        "\nFor each sample:"
        "\n  1. Original image appears."
        "\n  2. Close the plot."
        "\n  3. Image + mask appears."
        "\n  4. Close the plot to continue."
    )

    for i, stem in enumerate(
        paired_stems,
        start=1
    ):

        image_path = image_map[stem]
        mask_path = mask_map[stem]

        print("\n" + "-" * 70)
        print(
            f"{split.upper()} | "
            f"{i}/{len(paired_stems)}"
        )

        print(f"Image : {image_path.name}")
        print(f"Mask  : {mask_path.name}")

        # ----------------------------------------------------
        # STEP 1: Original image
        # ----------------------------------------------------

        show_original(
            image_path,
            f"{split.upper()} | {i}/{len(paired_stems)}\n"
            f"Original Image"
        )

        # ----------------------------------------------------
        # STEP 2: Image + mask
        # ----------------------------------------------------

        show_mask_verification(
            image_path,
            mask_path,
            f"{split.upper()} | {i}/{len(paired_stems)}\n"
            f"Lesion Mask Verification"
        )

    print(
        f"\nFinished {split.upper()} inspection."
    )


# ============================================================
# MAIN
# ============================================================

def main():

    for split in SPLITS:
        inspect_split(split)

    print("\n" + "=" * 70)
    print("ALL DATASET SPLITS INSPECTED")
    print("=" * 70)


if __name__ == "__main__":
    main()

# **Cell 1 — Mount + paths**

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/Potato_Severity_Dataset")
MANIFEST = ROOT / "manifest.csv"

print("Dataset:", ROOT)
print("Manifest exists:", MANIFEST.exists())
print("Root exists:", ROOT.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/Potato_Severity_Dataset
Manifest exists: True
Root exists: True


# **Cell 2 — Full automated QC**

In [ ]:
# ============================================================
# POTATO DATASET - AUTOMATED QC
# ============================================================

manifest = pd.read_csv(MANIFEST)

required_cols = {
    "file_name", "width", "height",
    "category_id", "class_name",
    "source_split", "final_split"
}

missing_cols = required_cols - set(manifest.columns)
if missing_cols:
    raise ValueError(f"Missing manifest columns: {missing_cols}")

splits = ["train", "valid", "test"]

results = []
missing_images = []
missing_masks = []
orphan_masks = []
duplicate_manifest = []

# ------------------------------------------------------------
# 1. Duplicate manifest entries
# ------------------------------------------------------------
dupes = manifest[manifest["file_name"].duplicated(keep=False)]

if len(dupes):
    duplicate_manifest = dupes["file_name"].tolist()

# ------------------------------------------------------------
# 2. Scan every split
# ------------------------------------------------------------
for split in splits:

    img_dir = ROOT / split / "images"
    mask_dir = ROOT / split / "lesion_masks"

    if not img_dir.exists():
        print(f"WARNING: image directory missing -> {img_dir}")
        continue

    if not mask_dir.exists():
        print(f"WARNING: mask directory missing -> {mask_dir}")
        continue

    image_files = {
        p.name: p for p in img_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    }

    mask_files = {
        p.name: p for p in mask_dir.iterdir()
        if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]
    }

    # --------------------------------------------------------
    # Manifest records belonging to this final split
    # --------------------------------------------------------
    split_df = manifest[manifest["final_split"] == split].copy()

    manifest_names = set(split_df["file_name"])

    # --------------------------------------------------------
    # Image / mask matching
    # --------------------------------------------------------
    for _, row in split_df.iterrows():

        fname = row["file_name"]
        stem = Path(fname).stem

        image_path = img_dir / fname

        # Mask expected with same stem, normally PNG
        candidate_masks = list(mask_dir.glob(stem + ".*"))
        mask_path = candidate_masks[0] if candidate_masks else None

        status = "OK"
        issues = []

        # -------------------------
        # Image existence
        # -------------------------
        if not image_path.exists():
            missing_images.append((split, fname))
            issues.append("missing_image")

            results.append({
                "split": split,
                "file_name": fname,
                "class_name": row["class_name"],
                "width_manifest": row["width"],
                "height_manifest": row["height"],
                "image_width": np.nan,
                "image_height": np.nan,
                "mask_pixels": np.nan,
                "mask_area_pct": np.nan,
                "status": "ERROR",
                "issues": ";".join(issues)
            })
            continue

        # -------------------------
        # Open image
        # -------------------------
        try:
            img = Image.open(image_path)
            iw, ih = img.size
        except Exception as e:
            issues.append(f"bad_image:{type(e).__name__}")
            results.append({
                "split": split,
                "file_name": fname,
                "class_name": row["class_name"],
                "width_manifest": row["width"],
                "height_manifest": row["height"],
                "image_width": np.nan,
                "image_height": np.nan,
                "mask_pixels": np.nan,
                "mask_area_pct": np.nan,
                "status": "ERROR",
                "issues": ";".join(issues)
            })
            continue

        # -------------------------
        # Manifest dimension check
        # -------------------------
        if iw != int(row["width"]) or ih != int(row["height"]):
            issues.append("manifest_dimension_mismatch")

        # -------------------------
        # Mask existence
        # -------------------------
        if mask_path is None:
            missing_masks.append((split, fname))
            issues.append("missing_mask")

            results.append({
                "split": split,
                "file_name": fname,
                "class_name": row["class_name"],
                "width_manifest": row["width"],
                "height_manifest": row["height"],
                "image_width": iw,
                "image_height": ih,
                "mask_pixels": np.nan,
                "mask_area_pct": np.nan,
                "status": "ERROR",
                "issues": ";".join(issues)
            })
            continue

        # -------------------------
        # Open mask
        # -------------------------
        try:
            mask = np.array(Image.open(mask_path).convert("L"))

            if mask.shape[1] != iw or mask.shape[0] != ih:
                issues.append("mask_dimension_mismatch")

            # Binary/non-binary diagnostic
            unique_values = np.unique(mask)

            # Any nonzero pixel considered lesion
            lesion = mask > 0
            mask_pixels = int(lesion.sum())
            total_pixels = mask.shape[0] * mask.shape[1]
            mask_area_pct = 100 * mask_pixels / total_pixels

        except Exception as e:
            issues.append(f"bad_mask:{type(e).__name__}")
            mask_pixels = np.nan
            mask_area_pct = np.nan
            unique_values = []

        # ----------------------------------------------------
        # Class ↔ mask consistency
        # ----------------------------------------------------
        class_name = str(row["class_name"]).strip().lower()

        if not np.isnan(mask_area_pct):

            if "healthy" in class_name:
                # Healthy should normally have empty lesion mask
                if mask_area_pct > 0.1:
                    issues.append("healthy_has_lesion_mask")

            elif "late blight" in class_name:
                # Late blight should normally have non-empty lesion mask
                if mask_area_pct == 0:
                    issues.append("late_blight_empty_mask")

        # ----------------------------------------------------
        # Extreme lesion-area warnings
        # ----------------------------------------------------
        if not np.isnan(mask_area_pct):

            if mask_area_pct > 80:
                issues.append("very_large_mask")

            elif "late blight" in class_name and mask_area_pct < 0.01:
                issues.append("very_small_lesion_mask")

        # ----------------------------------------------------
        # Determine final status
        # ----------------------------------------------------
        if any(x.startswith("missing_") for x in issues):
            status = "ERROR"
        elif any(
            x in issues for x in [
                "manifest_dimension_mismatch",
                "mask_dimension_mismatch",
                "healthy_has_lesion_mask",
                "late_blight_empty_mask",
                "very_large_mask",
                "very_small_lesion_mask"
            ]
        ):
            status = "WARNING"

        results.append({
            "split": split,
            "file_name": fname,
            "class_name": row["class_name"],
            "width_manifest": row["width"],
            "height_manifest": row["height"],
            "image_width": iw,
            "image_height": ih,
            "mask_pixels": mask_pixels,
            "mask_area_pct": round(mask_area_pct, 4)
                if not np.isnan(mask_area_pct) else np.nan,
            "status": status,
            "issues": ";".join(issues) if issues else ""
        })

    # --------------------------------------------------------
    # Find orphan masks
    # --------------------------------------------------------
    image_stems = {Path(x).stem for x in image_files.keys()}

    for mask_name, mask_path in mask_files.items():
        if Path(mask_name).stem not in image_stems:
            orphan_masks.append((split, mask_name))


# ------------------------------------------------------------
# Save report
# ------------------------------------------------------------
qc = pd.DataFrame(results)

report_path = ROOT / "dataset_qc_report.csv"
qc.to_csv(report_path, index=False)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\n================ QC SUMMARY ================\n")

print("Manifest records :", len(manifest))
print("QC records       :", len(qc))

print("\nStatus:")
print(qc["status"].value_counts(dropna=False))

print("\nMissing images :", len(missing_images))
print("Missing masks  :", len(missing_masks))
print("Orphan masks   :", len(orphan_masks))
print("Duplicate manifest rows :", len(duplicate_manifest))

print("\nClass distribution:")
print(manifest["class_name"].value_counts())

print("\nSplit distribution:")
print(manifest["final_split"].value_counts())

print("\nReport saved to:")
print(report_path)


================ QC SUMMARY ================

Manifest records : 1158
QC records       : 1158

Status:
status
OK    1158
Name: count, dtype: int64

Missing images : 0
Missing masks  : 0
Orphan masks   : 0
Duplicate manifest rows : 0

Class distribution:
class_name
Potato Healthy        591
Potato Late Blight    567
Name: count, dtype: int64

Split distribution:
final_split
train    926
valid    116
test     116
Name: count, dtype: int64

Report saved to:
/content/drive/MyDrive/Potato_Severity_Dataset/dataset_qc_report.csv


# **Cell 3 — শুধু suspicious cases দেখাবে**

In [ ]:
# ============================================================
# REVIEW SUSPICIOUS CASES
# ============================================================

pd.set_option("display.max_colwidth", 80)

suspicious = qc[qc["status"] != "OK"].copy()

print(f"Suspicious samples: {len(suspicious)}")

if len(suspicious):
    display(
        suspicious[
            [
                "split",
                "file_name",
                "class_name",
                "mask_area_pct",
                "status",
                "issues"
            ]
        ].sort_values(["status", "split"])
    )
else:
    print("✅ No suspicious samples found.")

# Also show lesion-area statistics for Late Blight
lb = qc[
    qc["class_name"].str.contains("Late Blight", case=False, na=False)
    & qc["mask_area_pct"].notna()
]

if len(lb):
    print("\nLate Blight lesion-area statistics:")
    display(
        lb["mask_area_pct"].describe(
            percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        ).to_frame()
    )

Suspicious samples: 0
✅ No suspicious samples found.

Late Blight lesion-area statistics:


,mask_area_pct
count,567.000000
mean,8.587332
std,6.911347
min,0.248700
1%,0.590980
5%,1.323890
25%,3.472100
50%,6.480400
75%,12.129950
95%,22.507050
